# Exercise 2: Hill's equation vs transfer matrices

**Block 1: Fields, kicks and linear optics**

**Learning objectives**
* Solve Hill's equation $x''+K(s)x=0$ numerically through a drift, a focusing and a defocusing quadrupole.
* See that the result is exactly a **matrix** times the initial ($x$, $x'$), and that the matrix has determinant 1.
* Understand *weak focusing* in a dipole: the $1/\rho^2$ term.

**Motivation.** In the literature Hill's equation is derived and solved piece by piece. Here
you let the computer solve the differential equation directly, and check that the transfer
matrices give the same answer much faster. This is why every accelerator code (MAD-X, Xsuite)
works with maps instead of solving differential equations.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

## Hill's equation

For a particle close to the reference orbit, the horizontal position *x*(*s*) obeys

$$x''(s) + K(s)\,x(s) = 0,$$

where ' means $\text{d}/\text{d}s$ and $K(s)$ is the focusing strength at position $s$:

| element | $K$ |
|---|---|
| drift | 0 |
| focusing quadrupole | $k$ > 0 [1/m$^2$] |
| defocusing quadrupole | −$k$ < 0 |
| sector dipole (bending radius $\rho$) | $1/\rho^2$ (horizontal plane) |

We write the state as $u=(x,x')$. Then $u'=(x',-Kx)$: two first-order equations,
which `scipy.integrate.solve_ivp` can solve.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from beamlib import elements as E, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()

## 1. Integrate through a beam line

A beam line is described as a list of segments `(length, K)`. Our test line is:
drift 1 m $\to$ focusing quad ($k$ = 1.2 $\text{m}^{-2}$, 0.5 m) → drift 1 m → defocusing quad
($k$ = −1.2 $\text{m}^{-2}$, 0.5 m) → drift 1 m.

**Task.** Write `hill_rhs` and `integrate_line`. Integrate segment by segment (so that $K$ is constant
inside each call to `solve_ivp`) and use tight tolerances (`rtol=1e-11, atol=1e-14`).

In [ ]:
line = [(1.0, 0.0), (0.5, 1.2), (1.0, 0.0), (0.5, -1.2), (1.0, 0.0)]   # (length [m], K [1/m^2])


def hill_rhs(s: float, u: np.ndarray, K: float) -> np.ndarray:
    """Right-hand side of Hill's equation: u = (x, x'), returns (x', x'')."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")


def integrate_line(segments, x0: float, xp0: float, n_points: int = 50):
    """Integrate Hill's equation through a list of (length, K) segments.

    Returns (s, x, xp): arrays with the trajectory sampled along the line
    (n_points per segment).  s starts at 0.
    """
    s_all, x_all, xp_all = [0.0], [x0], [xp0]
    u = np.array([x0, xp0])
    s0 = 0.0
    for length, K in segments:
        # YOUR CODE HERE
        raise NotImplementedError("Replace this line with your code")
    return np.array(s_all), np.array(x_all), np.array(xp_all)

In [ ]:
s, x, xp = integrate_line([(2.0, 0.0)], 1e-3, 2e-3)
assert np.isclose(x[-1], 1e-3 + 2.0 * 2e-3, rtol=1e-9), "a drift should give x = x0 + L x'0"
s, x, xp = integrate_line([(np.pi, 1.0)], 1e-3, 0.0)
assert np.isclose(x[-1], -1e-3, rtol=1e-8), "K = 1 for a length pi is half an oscillation"
print("GOOD! integrate_line works")

## 2. The matrix hidden in the solution

Because Hill's equation is **linear**, the final state is a linear function of the initial
state: $(x,x')_\text{end}=M (x,x')_\text{start}$.
The first column of $M$ is the trajectory that starts at (1, 0) (the "cosine-like" ray),
the second column is the one starting at (0, 1) (the "sine-like" ray).

**Task.** Build $M$ from two integrations, and build the same matrix by multiplying the
element matrices `E.drift`, `E.thick_quad`. Remember: the first element acts first, so it
stands on the **right** of the product.

In [ ]:
def matrix_from_integration(segments) -> np.ndarray:
    """2x2 transfer matrix obtained by integrating two special rays."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")


def matrix_from_elements(segments) -> np.ndarray:
    """2x2 transfer matrix as a product of element matrices."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
M_int = matrix_from_integration(line)
M_mat = matrix_from_elements(line)
print("from integration:\n", M_int)
print("from matrices:\n", M_mat)
print("det M =", np.linalg.det(M_mat))
assert np.allclose(M_int, M_mat, atol=1e-9), "the two matrices should agree"
assert abs(np.linalg.det(M_mat) - 1) < 1e-12, "det M must be 1"
print("GOOD! Integration and matrices agree, and det M = 1")

**Why is det *M* = 1?** The determinant is the factor by which areas in the $(x,x')$ plane
are stretched. Hill's equation has no damping/friction term (no $x'$ term), so phase-space areas are
conserved (Liouville's theorem). This is called **symplecticity**, and every correct map must have it.

## 3. Trajectories along the line

**Predict first.** Three particles start parallel to the axis ($x'$ = 0) at $x$ = 1, 2, 3 mm.
Will they be closer together or further apart after the focusing quad? After the whole line?

*Your prediction:*

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
edges = np.cumsum([0] + [seg[0] for seg in line])
for (L_, K), a, b in zip(line, edges[:-1], edges[1:]):
    if K != 0:
        ax.axvspan(a, b, color=COLORS["x"] if K > 0 else COLORS["y"], alpha=0.15)
for x0 in [1e-3, 2e-3, 3e-3]:
    s, x, xp = integrate_line(line, x0, 0.0)
    ax.plot(s, x * MM, label=rf"$x_0$ = {x0 * MM:.0f} mm (solve_ivp)")
    # matrix propagation, element by element
    u, xs = np.array([x0, 0.0]), [x0]
    for L_, K in line:
        u = E.thick_quad(K, L_) @ u
        xs.append(u[0])
    ax.plot(edges, np.array(xs) * MM, "ko", ms=4)
ax.plot([], [], "ko", label="matrices at element ends")
ax.set_xlabel(r"$s$ [m]"); ax.set_ylabel(r"$x$ [mm]")
ax.set_title("Trajectories (shaded: blue = focusing quad, red = defocusing quad)")
ax.legend(); plt.show()

## 4. Weak focusing in a dipole

A particle in a sector dipole moves on a circle of radius $\rho$. A particle that starts a bit off
the reference circle moves on a *neighbouring* circle, which crosses the reference circle
again after half a turn. Seen from the reference orbit this is an oscillation:
the dipole focuses with $K=1/\rho^2$. This is called **weak focusing**.

**Task.** Integrate through a dipole with $\rho=$ 4 m and length 1.6 m and compare with
`E.sector_dipole(L, rho)`. Then answer: what is the matrix of a dipole bending by a *full
circle* (length $2\pi\rho$)? Predict first, then check.

*Your prediction for the full circle:*

In [ ]:
rho, Lb = 4.0, 1.6
# YOUR CODE HERE
raise NotImplementedError("Replace this line with your code")
assert np.allclose(M_dip_int, E.sector_dipole(Lb, rho), atol=1e-9)
assert np.allclose(M_circle, np.eye(2), atol=1e-8), "a full circle brings every particle back"
print("dipole matrix:\n", M_dip_int)
print("full circle:\n", M_circle.round(10))
print("GOOD! Weak focusing understood")

## Reflection questions

1. Hill's equation is linear in $x$. Which of the magnets of Exercise 1 would break this
   linearity, and what would happen to the "matrix" idea then?
2. Why do we multiply matrices from right to left when the beam goes from left to right?
3. A quadrupole focuses with $K\approx1\text{ m}^{-2}$; the dipole here gives $1/\rho^2\approx0.06 \text{ m}^{-2}$.
   Why is the dipole focusing called *weak*, and why do modern rings use quadrupoles?